# Fatigue modeling

Ordinal and classification models with participant-level held-out test, GroupKFold CV, and Optuna tuning. Core logic lives in `src/modeling/`.

In [16]:
%pip install -q -r ../../requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [21]:
import sys
from pathlib import Path

_src = Path('../../src').resolve()
if str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

# Ensure local src edits are picked up when re-running this cell.
for _mod in [k for k in list(sys.modules) if k == 'modeling' or k.startswith('modeling.')]:
    del sys.modules[_mod]

import pandas as pd
from modeling.baselines import (
    add_delta_vs_baseline,
    run_all_baseline_benchmarks,
    summarize_baseline_metrics,
)
from modeling.config import DATA_PATH, HIGH_FATIGUE_THRESHOLD, N_CV_FOLDS, OPTUNA_TRIALS
from modeling.cv import run_model_benchmark
from modeling.data import load_fatigue_data, prepare_splits, split_summary_table
from modeling.registry import (
    CLASSIFICATION_MODELS,
    ORDINAL_MODELS,
    get_search_space,
    make_model_factory,
    resolve_training_data,
)
from modeling.tuning import tune_model


## 1. Load data and split

Participants are held out with a **stratified split** on per-participant high-fatigue rate so train/val and test have similar class balance.

“We randomly assign whole participants to train/val or test, but we do it in a way that both groups contain a similar proportion of people who often report high fatigue — not just a random 8 people who might all happen to be high-fatigue reporters.”


In [22]:
df = load_fatigue_data('../../' + DATA_PATH)
bundle = prepare_splits(df)
y_high_fatigue = (df['fatigue_num'] >= HIGH_FATIGUE_THRESHOLD).astype(int)

print(f"Rows: {len(df):,}  Participants: {df['id'].nunique()}")
display(split_summary_table(bundle, y_high_fatigue))
print('Test participant ids:', sorted(bundle.test_ids))


Rows: 3,331  Participants: 42


,split,participants,rows,mean_fatigue,high_fatigue_rate
0,train_val,34,2646,2.419123,0.267196
1,test,8,685,2.816058,0.319708


Test participant ids: [np.int64(10), np.int64(18), np.int64(30), np.int64(37), np.int64(38), np.int64(42), np.int64(46), np.int64(50)]


## 1b. Baseline benchmarks

Simple predictors evaluated with the same GroupKFold CV and held-out test protocol as the tuned models. Includes persistence baselines (`lag1_fatigue`, `expanding_mean`).

In [23]:
ordinal_baseline_results = run_all_baseline_benchmarks(bundle, task='ordinal', n_splits=N_CV_FOLDS)
classification_baseline_results = run_all_baseline_benchmarks(bundle, task='classification', n_splits=N_CV_FOLDS)

ordinal_baseline_summary = summarize_baseline_metrics(ordinal_baseline_results, task='ordinal')
clf_baseline_summary = summarize_baseline_metrics(classification_baseline_results, task='classification')

print('Ordinal baselines (test metrics)')
display(ordinal_baseline_summary[[c for c in ordinal_baseline_summary.columns if c.startswith('test_')]])
print('Classification baselines (test metrics)')
display(clf_baseline_summary[[c for c in clf_baseline_summary.columns if c.startswith('test_')]])

Ordinal baselines (test metrics)


,test_mae,test_rmse,test_r2,test_qwk
model,,,,
global_mean,1.321168,1.585979,-0.360095,0.000000
global_mode,0.986861,1.372302,-0.018295,0.000000
lag1_fatigue,0.931387,1.358402,0.002229,0.498804
expanding_mean,0.836496,1.162052,0.269827,0.463108


Classification baselines (test metrics)


,test_accuracy,test_f1,test_precision,test_recall
model,,,,
majority_class,0.680292,0.0,0.0,0.0


MAE: Mean Absolute Error;

RMSE: Root Mean Squared Error, measures the variation in residual/error

R2: how much variability is explained by the model

QWK: Quadratic Weighted Kappa. QWK measures the agreement between two raters—such as an AI and a human—on an ordered scale. It is designed to adjust for chance agreements and heavily penalize larger scoring discrepancies over minor ones.

## 2. Hyperparameter tuning

Use the Optuna framework. Use train/val dataset only. Held-out test ids are excluded from CV folds.

Each model gets 30 trials; objective is mean CV MAE (ordinal) or F1 (classification).

In [24]:
ordinal_best_params = {}
ordinal_best_cv = {}

for name in ORDINAL_MODELS:
    data_kwargs = resolve_training_data(name, bundle, task='ordinal')
    params, score = tune_model(
        name=name,
        registry=ORDINAL_MODELS,
        search_space=get_search_space(name),
        task='ordinal',
        n_trials=OPTUNA_TRIALS,
        n_splits=N_CV_FOLDS,
        test_ids=bundle.test_ids,
        **{k: v for k, v in data_kwargs.items() if k in {
            'X_train_val', 'y_train_val', 'groups', 'seq_train_val'
        }},
    )
    ordinal_best_params[name] = params
    ordinal_best_cv[name] = score
    print(f'{name}: best CV MAE = {score:.4f}  params = {params}')


ordered_logistic: best CV MAE = 1.4736  params = {'alpha': 9.98879940965251}
ordinal_rf: best CV MAE = 1.2597  params = {'n_estimators': 216, 'max_depth': 5, 'min_samples_leaf': 6}
catboost_ordinal: best CV MAE = 1.2444  params = {'iterations': 445, 'depth': 8, 'learning_rate': 0.01950377250870862, 'l2_leaf_reg': 8.925881007209712}
mixed_effects: best CV MAE = 1.4186  params = {'maxiter': 323}
lstm: best CV MAE = 1.5317  params = {'rnn_type': 'gru', 'hidden_size': 88, 'num_layers': 1, 'dropout': 0.23647067502029812, 'lr': 0.00010684006504904064, 'batch_size': 64, 'epochs': 40, 'patience': 10}


In [25]:
classification_best_params = {}
classification_best_cv = {}

for name in CLASSIFICATION_MODELS:
    data_kwargs = resolve_training_data(name, bundle, task='classification')
    params, score = tune_model(
        name=name,
        registry=CLASSIFICATION_MODELS,
        search_space=get_search_space(name),
        task='classification',
        n_trials=OPTUNA_TRIALS,
        n_splits=N_CV_FOLDS,
        test_ids=bundle.test_ids,
        **{k: v for k, v in data_kwargs.items() if k in {
            'X_train_val', 'y_train_val', 'groups'
        }},
    )
    classification_best_params[name] = params
    classification_best_cv[name] = score
    print(f'{name}: best CV F1 = {score:.4f}  params = {params}')


lightgbm: best CV F1 = 0.3142  params = {'n_estimators': 210, 'max_depth': 3, 'learning_rate': 0.14130012416861534, 'num_leaves': 25, 'min_child_samples': 18}
random_forest: best CV F1 = 0.3309  params = {'n_estimators': 282, 'max_depth': 3, 'min_samples_leaf': 6, 'max_features': 'log2'}


## 3. Final benchmarks (tuned params)

Refit on full train/val, evaluate once on held-out test.

In [26]:
ordinal_results = []
for name, params in ordinal_best_params.items():
    data_kwargs = resolve_training_data(name, bundle, task='ordinal')
    factory = make_model_factory(name, params, ORDINAL_MODELS)
    result = run_model_benchmark(
        name=name,
        model_factory=factory,
        task='ordinal',
        n_splits=N_CV_FOLDS,
        test_ids=bundle.test_ids,
        y_train_val=data_kwargs['y_train_val'],
        y_test=data_kwargs['y_test'],
        groups=data_kwargs['groups'],
        use_sequences=data_kwargs.get('use_sequences', False),
        **{k: v for k, v in data_kwargs.items() if k in {'X_train_val', 'X_test', 'seq_train_val', 'seq_test'}},
    )
    result['best_params'] = params
    ordinal_results.append(result)
    print(f'[ok] {name}')


[ok] ordered_logistic
[ok] ordinal_rf
[ok] catboost_ordinal
[ok] mixed_effects
[ok] lstm


In [27]:
classification_results = []
for name, params in classification_best_params.items():
    data_kwargs = resolve_training_data(name, bundle, task='classification')
    factory = make_model_factory(name, params, CLASSIFICATION_MODELS)
    result = run_model_benchmark(
        name=name,
        model_factory=factory,
        task='classification',
        n_splits=N_CV_FOLDS,
        test_ids=bundle.test_ids,
        y_train_val=data_kwargs['y_train_val'],
        y_test=data_kwargs['y_test'],
        groups=data_kwargs['groups'],
        **{k: v for k, v in data_kwargs.items() if k in {'X_train_val', 'X_test'}},
    )
    result['best_params'] = params
    classification_results.append(result)
    print(f'[ok] {name}')


[ok] lightgbm
[ok] random_forest


## 4. Results summary

In [28]:
def collect_summaries(results, task='ordinal'):
    cv_rows, test_rows = [], []
    metric_cols = ['mae', 'rmse', 'r2', 'qwk'] if task == 'ordinal' else ['accuracy', 'f1', 'precision', 'recall']
    for result in results:
        cv_mean = result['cv_summary'].loc['mean', metric_cols]
        cv_std = result['cv_summary'].loc['std', metric_cols]
        cv_row = {
            'model': result['name'],
            'best_params': str(result.get('best_params', {})),
        }
        test_row = {
            'model': result['name'],
            'best_params': str(result.get('best_params', {})),
        }
        for col in metric_cols:
            cv_row[f'cv_{col}'] = cv_mean[col]
            cv_row[f'cv_{col}_std'] = cv_std[col]
            test_row[f'test_{col}'] = result['test_metrics'][col]
        cv_rows.append(cv_row)
        test_rows.append(test_row)
    return pd.DataFrame(cv_rows).set_index('model'), pd.DataFrame(test_rows).set_index('model')

all_ordinal_results = ordinal_baseline_results + ordinal_results
all_classification_results = classification_baseline_results + classification_results

ordinal_cv_summary, ordinal_test_summary = collect_summaries(all_ordinal_results, task='ordinal')
clf_cv_summary, clf_test_summary = collect_summaries(all_classification_results, task='classification')

ordinal_test_summary = add_delta_vs_baseline(
    ordinal_test_summary,
    baseline_name='lag1_fatigue',
    metric_cols=['mae', 'rmse', 'r2', 'qwk'],
    task='ordinal',
)
clf_test_summary = add_delta_vs_baseline(
    clf_test_summary,
    baseline_name='majority_class',
    metric_cols=['accuracy', 'f1', 'precision', 'recall'],
    task='classification',
)

print('Ordinal CV summary (baselines first)')
display(ordinal_cv_summary)
print('Ordinal held-out test summary (delta vs lag1_fatigue baseline)')
display(ordinal_test_summary)
print('Classification CV summary (baselines first)')
display(clf_cv_summary)
print('Classification held-out test summary (delta vs majority_class)')
display(clf_test_summary)


Ordinal CV summary (baselines first)


,best_params,cv_mae,cv_mae_std,cv_rmse,cv_rmse_std,cv_r2,cv_r2_std,cv_qwk,cv_qwk_std
model,,,,,,,,,
global_mean,{},1.317778,0.057900,1.552164,0.046211,-0.088215,0.056688,0.000000,0.000000
global_mode,{},1.257337,0.147267,1.601647,0.088058,-0.157108,0.070130,0.000000,0.000000
lag1_fatigue,{},0.829233,0.132826,1.339758,0.168053,0.179388,0.207910,0.587082,0.105202
expanding_mean,{},0.916134,0.111049,1.246354,0.122451,0.298481,0.097713,0.519738,0.095469
ordered_logistic,{'alpha': 9.98879940965251},1.473643,0.177943,1.773626,0.115951,-0.423510,0.167278,-0.006894,0.097963
ordinal_rf,"{'n_estimators': 216, 'max_depth': 5, 'min_sam...",1.259746,0.041221,1.565079,0.050454,-0.110949,0.128198,0.102875,0.131480
catboost_ordinal,"{'iterations': 445, 'depth': 8, 'learning_rate...",1.244445,0.087018,1.504097,0.064566,-0.021607,0.065396,0.103609,0.060479
mixed_effects,{'maxiter': 323},1.418602,0.154267,1.799888,0.157781,-0.471656,0.259753,-0.049980,0.050471
lstm,"{'rnn_type': 'gru', 'hidden_size': 88, 'num_la...",1.531712,0.213315,1.900080,0.238541,-0.651408,0.415510,-0.023809,0.174296


Ordinal held-out test summary (delta vs lag1_fatigue baseline)


,best_params,test_mae,test_rmse,test_r2,test_qwk,delta_mae_vs_lag1_fatigue,delta_rmse_vs_lag1_fatigue,delta_r2_vs_lag1_fatigue,delta_qwk_vs_lag1_fatigue
model,,,,,,,,,
global_mean,{},1.321168,1.585979,-0.360095,0.000000,0.389781,0.227577,-0.362324,-0.498804
global_mode,{},0.986861,1.372302,-0.018295,0.000000,0.055474,0.013900,-0.020524,-0.498804
lag1_fatigue,{},0.931387,1.358402,0.002229,0.498804,0.000000,0.000000,0.000000,0.000000
expanding_mean,{},0.836496,1.162052,0.269827,0.463108,-0.094891,-0.196350,0.267598,-0.035696
ordered_logistic,{'alpha': 9.98879940965251},1.272993,1.613809,-0.408247,-0.099962,0.341606,0.255407,-0.410476,-0.598766
ordinal_rf,"{'n_estimators': 216, 'max_depth': 5, 'min_sam...",1.077372,1.456924,-0.147753,-0.019176,0.145985,0.098522,-0.149981,-0.517980
catboost_ordinal,"{'iterations': 445, 'depth': 8, 'learning_rate...",1.074453,1.432674,-0.109863,-0.028924,0.143066,0.074272,-0.112091,-0.527728
mixed_effects,{'maxiter': 323},1.211679,1.680285,-0.526653,-0.043040,0.280292,0.321883,-0.528882,-0.541844
lstm,"{'rnn_type': 'gru', 'hidden_size': 88, 'num_la...",1.373723,1.677241,-0.521128,-0.081051,0.442336,0.318840,-0.523356,-0.579855


Classification CV summary (baselines first)


,best_params,cv_accuracy,cv_accuracy_std,cv_f1,cv_f1_std,cv_precision,cv_precision_std,cv_recall,cv_recall_std
model,,,,,,,,,
majority_class,{},0.732127,0.032441,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
lightgbm,"{'n_estimators': 210, 'max_depth': 3, 'learnin...",0.614509,0.130911,0.314243,0.032802,0.351028,0.116646,0.339198,0.154725
random_forest,"{'n_estimators': 282, 'max_depth': 3, 'min_sam...",0.566408,0.126642,0.330884,0.125420,0.315641,0.168310,0.425654,0.279778


Classification held-out test summary (delta vs majority_class)


,best_params,test_accuracy,test_f1,test_precision,test_recall,delta_accuracy_vs_majority_class,delta_f1_vs_majority_class,delta_precision_vs_majority_class,delta_recall_vs_majority_class
model,,,,,,,,,
majority_class,{},0.680292,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
lightgbm,"{'n_estimators': 210, 'max_depth': 3, 'learnin...",0.385401,0.404526,0.293033,0.652968,0.294891,-0.404526,-0.293033,-0.652968
random_forest,"{'n_estimators': 282, 'max_depth': 3, 'min_sam...",0.318248,0.448642,0.302548,0.867580,0.362044,-0.448642,-0.302548,-0.867580


### Notes

- Train/test split is **stratified by participant high-fatigue rate** (`prepare_splits(..., stratify=True)`).
- **Baselines** include persistence models: `lag1_fatigue` (yesterday's fatigue) and `expanding_mean` (mean of prior days).
- Test summary includes **delta vs `lag1_fatigue`** (negative `delta_mae` = better than persistence).
- Compare **LSTM** to the same `lag1_fatigue` / `expanding_mean` rows (sequence-path duplicates were removed).
- **mixed_effects** is a population ordinal model on day-varying features (no participant dummies, so it generalizes to held-out ids).
- **lstm** uses feature scaling, class-weighted loss, gradient clipping, and participant-level early stopping.
- Tuning uses train/val only; test participants never appear in CV folds.